# LDA topics & entropy per sliding window

Loads the **trained LDA model** and the **VQ-VAE profiles PKL**, then for each patient:

1. Rebuilds the day-type sequence (`embedding_ids`).
2. Applies the LDA model on **sliding windows** (default 30 days).
3. Extracts the topic distribution per window.
4. Computes the **Shannon entropy** per window and plots it.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim import corpora
from gensim.models import LdaModel

# --- Paths (relative to repo root) ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

LDA_MODEL_PATH = PROJECT_ROOT / "data/processed/lda/lda_model_6topics.gensim"
DICTIONARY_PATH = PROJECT_ROOT / "data/processed/lda/dictionary_lda_180patients.dict"
PROFILES_PKL = PROJECT_ROOT / "data/processed/vq-vae/profiles_per_sample_oncology_28_12_2025.pkl"

# VQ-VAE variant / codebook size used to train the LDA model
MODEL_TYPE = "a0"
N = 30
WINDOW_SIZE = 30   # days per window
STEP = 1           # sliding step (1 = day-by-day)
MIN_EMBEDDINGS = 15

print("Model :", LDA_MODEL_PATH)
print("Dict  :", DICTIONARY_PATH)
print("PKL   :", PROFILES_PKL)

## 1 · Load the trained LDA model and dictionary

In [ ]:
lda_model = LdaModel.load(str(LDA_MODEL_PATH))
dictionary = corpora.Dictionary.load(str(DICTIONARY_PATH))

NUM_TOPICS = lda_model.num_topics
print(f"LDA topics: {NUM_TOPICS} | dictionary size: {len(dictionary)}")

## 2 · Load the VQ-VAE PKL and rebuild per-patient day-type sequences

Structure: `pkl[MODEL_TYPE][N][patient_id][0][1]` is the array of `embedding_ids`.

In [ ]:
with open(PROFILES_PKL, "rb") as f:
    profiles = pickle.load(f)

users = profiles[MODEL_TYPE][N]

rows = []
for patient_id, payload in users.items():
    embedding_ids = np.asarray(payload[0][1]).astype(int).tolist()
    rows.append({"id": patient_id, "embedding_ids": embedding_ids})

df = pd.DataFrame(rows).sort_values("id").reset_index(drop=True)
df["n_days"] = df["embedding_ids"].apply(len)
print(f"Patients: {len(df)} | days range: {df['n_days'].min()}–{df['n_days'].max()}")
df.head()

## 3 · Apply LDA per sliding window + Shannon entropy

In [ ]:
def topic_distributions_sliding(embedding_ids, window_size=WINDOW_SIZE, step=STEP):
    """Topic distribution (length = NUM_TOPICS) for every sliding window."""
    n = len(embedding_ids)
    if n < MIN_EMBEDDINGS:
        return np.empty((0, NUM_TOPICS))

    dists = []
    if n < window_size:
        windows = [embedding_ids]
    else:
        windows = [embedding_ids[s:s + window_size]
                   for s in range(0, n - window_size + 1, step)]

    for window in windows:
        bow = dictionary.doc2bow([str(tok) for tok in window])
        topics = lda_model.get_document_topics(bow, minimum_probability=0.0)
        probs = np.zeros(NUM_TOPICS)
        for tid, p in topics:
            probs[tid] = p
        dists.append(probs)
    return np.array(dists)


def shannon_entropy(dists, normalized=True):
    """Per-window Shannon entropy. If normalized, divide by log(NUM_TOPICS) → [0, 1]."""
    if len(dists) == 0:
        return np.empty(0)
    p = np.clip(dists, 1e-12, 1.0)
    ent = -np.sum(p * np.log(p), axis=1)
    if normalized:
        ent = ent / np.log(NUM_TOPICS)
    return ent

## 4 · Single patient: topic evolution + entropy

In [ ]:
patient_id = df.iloc[0]["id"]   # change to any id in df['id']

emb = df.loc[df["id"] == patient_id, "embedding_ids"].iloc[0]
dists = topic_distributions_sliding(emb)
ent = shannon_entropy(dists, normalized=True)
print(f"Patient {patient_id}: {len(dists)} windows")

fig, axs = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for k in range(NUM_TOPICS):
    axs[0].plot(dists[:, k], label=f"Topic {k}")
axs[0].set_ylabel("Topic probability")
axs[0].set_title(f"Topic evolution - patient {patient_id}")
axs[0].legend(loc="upper right", ncol=NUM_TOPICS, fontsize="small")
axs[0].grid(alpha=0.3)

axs[1].plot(ent, color="black")
axs[1].set_ylabel("Normalized entropy")
axs[1].set_xlabel("Window (sliding day index)")
axs[1].set_title("Behavioral entropy per window")
axs[1].set_ylim(0, 1)
axs[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5 · Entropy summary across all patients (optional)

In [ ]:
summary = []
for _, r in df.iterrows():
    e = shannon_entropy(topic_distributions_sliding(r["embedding_ids"]), normalized=True)
    if len(e) == 0:
        continue
    summary.append({
        "id": r["id"],
        "n_windows": len(e),
        "entropy_mean": float(np.mean(e)),
        "entropy_std": float(np.std(e)),
        "entropy_min": float(np.min(e)),
        "entropy_max": float(np.max(e)),
    })

entropy_df = pd.DataFrame(summary)
print(f"Patients with windows: {len(entropy_df)}")
entropy_df.head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(entropy_df["entropy_mean"], bins=20, color="steelblue", edgecolor="white")
plt.xlabel("Mean normalized entropy per patient")
plt.ylabel("Number of patients")
plt.title("Distribution of behavioral entropy")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()